# Fine-tuning a Medical Q&A Assistant with QLoRA (Mistral-7B-Instruct-v0.2)

**Runtime required: GPU (T4 is fine).** In Colab: `Runtime > Change runtime type > T4 GPU`.

This notebook drives the scripts in `src/` end-to-end:
1. `prepare_dataset.py` — download + clean the medical Q&A dataset, format it for Mistral's instruct template
2. `finetune.py` — QLoRA fine-tuning (4-bit base model + LoRA adapter)
3. `evaluate.py` — base vs. fine-tuned comparison on held-out questions (ROUGE-L + qualitative examples)
4. `inference.py` — ask the fine-tuned model a question interactively

See `docs/theory.md` for the quantization/LoRA/QLoRA math behind steps 1-2, and the top-level `README.md` for the full project write-up.

**Disclaimer:** this is a fine-tuning skills demo on a small open medical Q&A dataset. It is *not* a clinically validated tool and must never be used for real medical decisions.

## 0. Get the project onto this Colab instance

Pick **one** of the two cells below depending on where this repo lives.

In [3]:
# Option A: clone from GitHub (recommended once you've pushed this repo)
# Replace the URL with your own fork/repo.
!git clone https://github.com/Kritikaagra/medical-qa-assistant.git
%cd medical-qa-assistant

Cloning into 'medical-qa-assistant'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 19 (delta 1), reused 19 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 18.45 KiB | 18.45 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/medical-qa-assistant


In [ ]:
# Option B: no GitHub repo yet — zip the project locally and upload it here instead.
# from google.colab import files
# uploaded = files.upload()  # upload finetuning-medical-qa-assistant.zip
# !unzip -q finetuning-medical-qa-assistant.zip
# %cd finetuning-medical-qa-assistant

## 1. Install dependencies

In [1]:
!pip3 install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 2. Hugging Face authentication

`mistralai/Mistral-7B-Instruct-v0.2` is ungated, so a token isn't strictly required to download it — but set one anyway to avoid rate limits, and because you'll want it if you swap in a gated model (e.g. Llama 2, Gemma) later.

Add your token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), then run:

In [ ]:
import os

from google.colab import userdata



os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")



from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

## 3. Prepare the dataset (no GPU needed for this step)

In [ ]:
!python src/prepare_dataset.py

## 4. QLoRA fine-tuning

On a free T4 with the default `config.py` settings (~3000 training examples, 1 epoch, batch size 4 x grad-accum 4), this takes roughly 20-40 minutes. Loss should be logged every 10 steps.

In [ ]:
!python src/finetune.py

## 5. Evaluate: base model vs. fine-tuned adapter

Generates answers from both the untouched base model and the fine-tuned adapter on the same held-out questions, scores them with ROUGE-L against the reference answers, and writes `results/eval_results.md` + `results/eval_raw.json`.

In [ ]:
!python src/evaluate.py

In [ ]:
from IPython.display import Markdown, display



with open("results/eval_results.md") as f:

    display(Markdown(f.read()))

## 6. Try it yourself

In [ ]:
import sys

sys.path.insert(0, "src")



import inference



model, tokenizer = inference.load_model_and_tokenizer()

print(inference.ask(model, tokenizer, "What are the early symptoms of anemia?"))

## Next steps

- Push this repo to GitHub and update the `git clone` URL in step 0.
- Try a higher LoRA rank (`LORA_R` in `src/config.py`) if you have more GPU time, and compare ROUGE-L / qualitative quality against this run.
- Swap in a different open base model (e.g. `google/gemma-2b-it`, gated — needs license acceptance on Hugging Face) by changing `BASE_MODEL_ID` and adjusting `LORA_TARGET_MODULES` for that architecture.
- See `docs/theory.md` for the quantization/LoRA math this pipeline relies on.